# UKB LOOK: Frozen-Classifier Incomplete-Modality Study

This notebook is the complete interface after Steps 1-14 produce a validated paired CFP-OCT cohort. The complete-modality classifier is frozen before filling, LOOK fitting, and evaluation. Classifier and paired-cGAN training use DDP when multiple GPUs are listed; LOOK and evaluation use the first listed GPU.


## 1. Configuration

Edit this cell and run all cells. Only `GPU_DEVICES` controls compute: `[0]` uses GPU 0, `[1]` uses GPU 1, and `[0, 1]` uses two-GPU DDP. Global batch sizes remain invariant. The old validated dataset is reused read-only while all cache and runs use this release timestamp.


In [ ]:
from pathlib import Path
import os

# Deployment. Optional overrides make the release portable to another Linux host.
PROJECT_ROOT = Path("/home/mengh/LOOK/2026_09_01_16_06_11")
DATA_ROOT_OVERRIDE = None
DATASET_ROOT_OVERRIDE = None
CACHE_ROOT_OVERRIDE = None
RUNS_ROOT_OVERRIDE = None
GPU_DEVICES = [0, 1]  # [0], [1], or [0, 1]; no other GPU setting is required
EXECUTION_MODE = "validation"  # dry_run, validation, freeze, or test
FROZEN_MANIFEST = None  # required only for test; produced by freeze
RESUME = True
RESTART = False
SMOKE_LIMIT = None
CHECK_ALL_IMAGE_PATHS = False
BOOTSTRAP_ITERATIONS = 2000

# Independent outer axes. Singleton lists run one configuration.
BACKBONES = ["resnet50"]
FUSION_POSITIONS = ["input", "stem", "layer1", "layer2", "layer3", "layer4", "feature"]
SEEDS = [3407, 3408, 3409]
FILLING_STRATEGIES = ["normalized_mean", "paired_cgan"]

# Complete-modality classifier profile. Batch sizes are global, not per GPU.
CLASSIFIER_PROFILES = [{
    "name": "primary", "epochs": 50, "patience": 10,
    "effective_batch_size": 32, "micro_batch_size": 32, "num_workers": 8,
    "pretrained_lr": 1e-4, "new_layer_lr": 1e-3, "weight_decay": 1e-4,
    "warmup_epochs": 5, "sampler_power": 0.5, "amp": True,
    "monitor_nodes": ["loss", "batch_accuracy", "fusion_logits"],
    "monitor_edges": ["fusion_classifier_edge"], "monitor_interval_steps": 50,
}]

# Independently trained paired-cGAN profile; ignored by normalized_mean arms.
GAN_PROFILES = [{
    "name": "primary", "gan_validation_fraction": 0.10,
    "gan_epochs": 100, "gan_patience": 10, "gan_batch_size": 16,
    "gan_num_workers": 8, "gan_learning_rate": 2e-4, "gan_beta1": 0.5,
    "gan_lambda_l1": 100.0, "gan_base_channels": 64,
}]

# LOOK validation-search profile. Singleton candidates define controlled ablations.
LOOK_PROFILES = [{
    "name": "primary", "enabled": True, "evaluate_random_missing": True,
    "missing_patterns": ["oct_missing", "cfp_missing"], "missing_ratios": [0.2, 0.4, 0.6, 0.8],
    "correction_nodes": ["all_available"], "downsample_factors": [4, 8, 16],
    "latent_dims": [16, 32, 64, 128, 256], "max_pca_rank": 256,
    "alpha_grid": [0.0, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0], "primary_metric": "macro_f1",
}]

os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(map(str, GPU_DEVICES))


## 2. Resolve release paths and compute topology

Step 3 registers the `LOOK` kernel. `project.json` centrally maps source/cache/runs to the new timestamp and the validated dataset to the preceding timestamp.


In [ ]:
import json
import torch
from look_core.distributed import parse_gpu_devices
from look_core.paths import ProjectPaths
from look_core.study_grid import StudyGrid, expand_study_grid, freeze_study_grid, run_study_grid

GPU_DEVICES = list(parse_gpu_devices(GPU_DEVICES))
if EXECUTION_MODE != "dry_run" and not torch.cuda.is_available(): raise RuntimeError("CUDA is required")
if torch.cuda.is_available() and torch.cuda.device_count() != len(GPU_DEVICES):
    raise RuntimeError(f"Requested {GPU_DEVICES}, but {torch.cuda.device_count()} devices are visible")
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
PATHS = ProjectPaths.load(project_root=PROJECT_ROOT, data_root=DATA_ROOT_OVERRIDE, dataset_root=DATASET_ROOT_OVERRIDE, cache_root=CACHE_ROOT_OVERRIDE, runs_root=RUNS_ROOT_OVERRIDE)
PATHS.ensure_runtime_layout()
print(json.dumps({"project_root":str(PATHS.project_root), "dataset_root_read_only":str(PATHS.dataset_root), "cache_root":str(PATHS.cache_root), "runs_root":str(PATHS.runs_root), "physical_gpu_devices":GPU_DEVICES, "world_size":len(GPU_DEVICES), "visible_gpu_names":[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}, indent=2))


## 3. Resolve and validate the study plan

Every list expands deterministically. Global classifier and cGAN batches are divided by world size; invalid divisibility fails before training.


In [ ]:
GRID = StudyGrid(backbones=BACKBONES, fusion_positions=FUSION_POSITIONS, seeds=SEEDS, filling_strategies=FILLING_STRATEGIES, classifier_profiles=CLASSIFIER_PROFILES, gan_profiles=GAN_PROFILES, look_profiles=LOOK_PROFILES)
PHASE = "test" if EXECUTION_MODE == "test" else "validation"
FROZEN_PATH = Path(FROZEN_MANIFEST) if FROZEN_MANIFEST else None
CASES = expand_study_grid(GRID, PATHS, phase=PHASE, frozen_manifest=FROZEN_PATH, resume=RESUME, restart=RESTART, smoke_limit=SMOKE_LIMIT, check_all_image_paths=CHECK_ALL_IMAGE_PATHS, bootstrap_iterations=BOOTSTRAP_ITERATIONS, gpu_devices=tuple(GPU_DEVICES))
print(f"Resolved configurations: {len(CASES)}")
for case in CASES[:10]: print(case.selection.experiment_id, "| world_size=", case.config.world_size, "| classifier batch/GPU=", case.config.per_device_micro_batch_size, "| cGAN batch/GPU=", case.config.per_device_gan_batch_size)
if len(CASES) > 10: print(f"... and {len(CASES)-10} more")


## 4. Run, resume, freeze, or test

`validation` may train and fit LOOK; `freeze` hashes selected artifacts; `test` is read-only with respect to those artifacts. Rank-zero epoch summaries appear below. Rerunning the same interrupted configuration resumes its checkpoints.


In [ ]:
if EXECUTION_MODE == "freeze":
    STUDY_RESULT = freeze_study_grid(GRID, PATHS, DEVICE, gpu_devices=tuple(GPU_DEVICES))
else:
    STUDY_RESULT = run_study_grid(GRID, PATHS, DEVICE, execute=EXECUTION_MODE != "dry_run", phase=PHASE, frozen_manifest=FROZEN_PATH, resume=RESUME, restart=RESTART, smoke_limit=SMOKE_LIMIT, check_all_image_paths=CHECK_ALL_IMAGE_PATHS, bootstrap_iterations=BOOTSTRAP_ITERATIONS, gpu_devices=tuple(GPU_DEVICES))
print(json.dumps(STUDY_RESULT, indent=2))


## 5. Inspect live validation monitors and results

The graph-internal `loss` node is differentiable. `batch_accuracy` and `fusion_logits` are online training diagnostics. After every epoch, rank zero prints validation loss, accuracy, balanced accuracy, macro/weighted F1, macro-AUROC, ECE, Brier score, and kappa; it also refreshes the main and per-class validation figures. Full JSONL records retain per-class metrics and confusion matrices even if training is interrupted.


In [ ]:
from IPython.display import Image as NotebookImage, display
print("Experiments:", PATHS.runs_root / "experiments")
print("Backbone monitors:", PATHS.runs_root / "backbones")
print("cGAN monitors:", PATHS.runs_root / "generators")
curves = sorted(PATHS.runs_root.glob("**/training_curves.png"), key=lambda p:p.stat().st_mtime)
if curves:
    print("Latest training curves:", curves[-1]); display(NotebookImage(filename=str(curves[-1])))
else:
    print("No completed curve yet; dry-run creates only a study plan.")
